# Deep Learning for NLP – Lab Exercise 1
## Biomedical Text Classification with PyTorch (LitCovid, BioCreative VII)

**Nombre:** _________________ &nbsp;&nbsp; **Fecha:** _________________

En este notebook se implementan, entrenan y comparan dos arquitecturas neuronales para la clasificación **multi-etiqueta** de artículos biomédicos sobre COVID-19 en 7 temas (Treatment, Diagnosis, Prevention, Mechanism, Transmission, Epidemic Forecasting, Case Report):

1. Un clasificador **Bag-of-Words (CBOW)** basado en embeddings (mean pooling).
2. Una **CNN** para texto con varias ventanas de convolución.

Estructura (sigue el enunciado):
0. Configuración y reproducibilidad
1. Dataset (carga y exploración)
2. Preprocesamiento (limpieza, vocabulario, ids, padding)
3. Tensores PyTorch y DataLoaders
4. Modelos (BOW y CNN)
5. Función de pérdida
6. Bucle de entrenamiento
7. Evaluación
8. Experimentos y comparación
9. Discusión
10. Extensiones opcionales

> Las respuestas que dependen de los números obtenidos están marcadas con **(revisar tras ejecutar)**: compruébalas con tus propios resultados antes de entregar.

## 0. Configuración y reproducibilidad

In [ ]:
# Ejecutar solo la primera vez (p. ej. en Colab)
# !pip install datasets scikit-learn matplotlib pandas

In [ ]:
import os, re, time, random, copy
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch as th
import torch.nn as nn
import torch.nn.functional as F
import torch.autograd as ag
from torch.utils.data import TensorDataset, DataLoader

from datasets import load_dataset
from sklearn.metrics import precision_score, recall_score, f1_score, classification_report

SEED = 42
def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    th.manual_seed(seed)
    th.cuda.manual_seed_all(seed)
    th.backends.cudnn.deterministic = True
    th.backends.cudnn.benchmark = False

set_seed()
device = th.device("cuda" if th.cuda.is_available() else "cpu")
print("Device:", device)

# Hiperparámetros globales
MAX_LEN    = 500     # longitud máxima de secuencia (sección 2.3)
BATCH_SIZE = 64
EPOCHS     = 5
LR         = 1e-3
QUICK_RUN  = False   # True = usar un subconjunto pequeño para probar el código rápidamente

## 1. Dataset
### 1.1 Cargar el dataset

In [ ]:
dataset = load_dataset("KushT/LitCovid_BioCreative")
print(dataset)

In [ ]:
print(dataset["train"][0])

**Nota sobre los splits.** En la versión de Hugging Face los nombres no coinciden con el enunciado: el split `test` contiene **6.239** artículos (= el *development set* oficial) y `validation` contiene **2.500** (= el tamaño del test oficial). Ambos incluyen etiquetas. Para seguir el enunciado (train 24.960 / dev 6.239) usamos como *dev* el split de **6.239** artículos, que se detecta automáticamente a continuación.

In [ ]:
for split in dataset:
    print(f"{split:12s}: {len(dataset[split]):,} artículos")

# El development set oficial tiene 6.239 artículos
DEV_KEY = "test" if len(dataset["test"]) == 6239 else "validation"
print("Usamos como development set el split:", DEV_KEY)

train_ds = dataset["train"]
dev_ds   = dataset[DEV_KEY]

if QUICK_RUN:
    train_ds = train_ds.select(range(3000))
    dev_ds   = dev_ds.select(range(1000))

print("Train:", len(train_ds), "| Dev:", len(dev_ds))

In [ ]:
ex = train_ds[0]
print("TITLE   :", ex["title"])
print("LABEL   :", ex["label"])
print("TEXT    :", ex["text"][:600], "...")

In [ ]:
labels = [
    "Treatment",
    "Diagnosis",
    "Prevention",
    "Mechanism",
    "Transmission",
    "Epidemic Forecasting",
    "Case Report"
]
NUM_LABELS = len(labels)

Y_train = np.array(train_ds["label"], dtype=np.float32)   # (N_train, 7)
Y_dev   = np.array(dev_ds["label"],   dtype=np.float32)   # (N_dev, 7)

n_labels_per_doc = Y_train.sum(axis=1)
print("Nº etiquetas por artículo (train):")
print(pd.Series(n_labels_per_doc).value_counts().sort_index().rename("artículos"))
print(f"\nMedia de etiquetas por artículo: {n_labels_per_doc.mean():.2f}")
print(f"Artículos con más de una etiqueta: {(n_labels_per_doc > 1).mean()*100:.1f}%")

**Respuestas 1.1**

1. **Artículos por split:** train = 24.960, development = 6.239 (split `test` en HF), test oficial = 2.500 (split `validation` en HF). Total 33.699.
2. **Campo `text`:** es la concatenación `'[Title]: ' + title + ' [Abstract]: ' + abstract + ' [Keywords]: ' + keywords`. Es decir, contiene título, resumen y palabras clave del autor, con marcadores de sección.
3. **Representación de las etiquetas:** un vector *multi-hot* de longitud 7 con 0/1, en el orden `Treatment, Diagnosis, Prevention, Mechanism, Transmission, Epidemic Forecasting, Case Report`. Por ejemplo `[1,0,0,1,0,0,0]` = Treatment + Mechanism.
4. **¿Varias etiquetas?** Sí. Es un problema multi-etiqueta: como se ve arriba, una parte importante de los artículos tiene 2 o más temas.
5. **Distribución:** ver sección 1.2.

### 1.2 Distribución de etiquetas

In [ ]:
# Número de ejemplos positivos por etiqueta
train_counts = Y_train.sum(axis=0).astype(int)
dev_counts   = Y_dev.sum(axis=0).astype(int)

label_df = pd.DataFrame({
    "train": train_counts,
    "train_%": (100 * train_counts / len(Y_train)).round(1),
    "dev": dev_counts,
    "dev_%": (100 * dev_counts / len(Y_dev)).round(1),
}, index=labels).sort_values("train", ascending=False)
label_df

In [ ]:
order = np.argsort(-train_counts)
fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.bar([labels[i] for i in order], train_counts[order], color="#4C72B0")
ax.bar_label(bars, fmt="%d", fontsize=9)
ax.set_ylabel("Nº de artículos")
ax.set_title("Distribución de temas en el conjunto de entrenamiento")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

**Respuestas 1.2** *(revisar tras ejecutar)*

1. **Más frecuente:** *Prevention* (≈ 45 % de los artículos de train), seguido de *Treatment* y *Diagnosis*.
2. **Menos frecuente:** *Epidemic Forecasting* (≈ 2–3 %), seguido de *Transmission*.
3. **¿Balanceado?** No. Hay un desequilibrio fuerte: la clase mayoritaria tiene ~20 veces más ejemplos positivos que la minoritaria.
4. **Problemas del desequilibrio:** la pérdida está dominada por las clases frecuentes, así que el modelo puede aprender a predecir casi siempre 0 en las clases raras y aun así obtener una pérdida baja y una *accuracy* alta. Resultado: recall bajo en las clases minoritarias, macro-F1 bajo, y el umbral de 0,5 deja de ser adecuado para esas clases (sus probabilidades rara vez lo superan). También hace que la evaluación con *accuracy* sea engañosa.

## 2. Preprocesamiento del texto

In [ ]:
def clean_str(string, tolower=True):
    # Basic text cleaning and tokenization.
    string = re.sub(r"[^A-Za-z0-9(),!?\'\`]", " ", string)
    string = re.sub(r"\s{2,}", " ", string)
    if tolower:
        string = string.lower()
    return string.strip()

def tokenize(text):
    return clean_str(text).split()

print(tokenize(train_ds[0]["text"])[:40])

In [ ]:
# Versiones tokenizadas de train y dev (usamos el campo `text`)
t0 = time.time()
train_tokens = [tokenize(t) for t in train_ds["text"]]
dev_tokens   = [tokenize(t) for t in dev_ds["text"]]
print(f"Tokenización: {time.time()-t0:.1f}s")
print("Ejemplo:", train_tokens[1][:25])

*Decisión de implementación:* los marcadores `[Title]:`, `[Abstract]:` y `[Keywords]:` se convierten en los tokens `title`, `abstract` y `keywords`. Aparecen en todos los artículos, así que no aportan información discriminativa pero tampoco perjudican; se mantienen para respetar el campo `text` tal como se pide.

### 2.1 Construcción del vocabulario

In [ ]:
word2id = {
    "<PAD>": 0,
    "<UNK>": 1
}

MIN_FREQ = 1   # el enunciado pide todas las palabras de train; subir a 2 reduce mucho el vocabulario

# Vocabulario construido SOLO con datos de entrenamiento
word_freq = Counter(tok for doc in train_tokens for tok in doc)
for word, freq in word_freq.most_common():
    if freq >= MIN_FREQ:
        word2id[word] = len(word2id)

id2word = {i: w for w, i in word2id.items()}
VOCAB_SIZE = len(word2id)
print("Tamaño del vocabulario:", VOCAB_SIZE)
print("Palabras más frecuentes:", word_freq.most_common(15))

dev_oov = sum(1 for doc in dev_tokens for t in doc if t not in word2id) / sum(len(d) for d in dev_tokens)
print(f"Tasa de tokens OOV (-> <UNK>) en dev: {dev_oov*100:.2f}%")

**¿Por qué no usar los datos de dev para construir el vocabulario?** El conjunto de desarrollo simula datos *no vistos*. Si el vocabulario incluyera palabras que solo aparecen en dev, se filtraría información de evaluación al modelo (*data leakage*) y la estimación del rendimiento sería optimista. Además, los embeddings de esas palabras nunca se entrenarían (no aparecen en train), así que no aportarían nada: en un escenario real los textos nuevos contendrán palabras desconocidas y el modelo debe aprender a manejarlas mediante `<UNK>`.

### 2.2 Convertir el texto en representaciones numéricas

In [ ]:
UNK_ID = word2id["<UNK>"]
PAD_ID = word2id["<PAD>"]

def text_to_ids(tokens, word2id):
    # Las palabras fuera del vocabulario se mapean a <UNK>
    return [word2id.get(tok, UNK_ID) for tok in tokens]

print(text_to_ids("covid treatment clinical study".split(), word2id))
print(text_to_ids("covid xyzunknownword".split(), word2id))

train_ids = [text_to_ids(doc, word2id) for doc in train_tokens]
dev_ids   = [text_to_ids(doc, word2id) for doc in dev_tokens]

### 2.3 Padding y truncado
Primero miramos la distribución de longitudes para justificar `MAX_LEN`.

In [ ]:
lengths = np.array([len(d) for d in train_ids])
print(f"Longitud media: {lengths.mean():.0f} | mediana: {np.median(lengths):.0f} | "
      f"p90: {np.percentile(lengths, 90):.0f} | p95: {np.percentile(lengths, 95):.0f} | máx: {lengths.max()}")
print(f"% de artículos truncados con MAX_LEN={MAX_LEN}: {(lengths > MAX_LEN).mean()*100:.1f}%")

plt.figure(figsize=(8, 3.5))
plt.hist(lengths, bins=80, range=(0, 1000), color="#4C72B0")
plt.axvline(MAX_LEN, color="red", ls="--", label=f"MAX_LEN = {MAX_LEN}")
plt.xlabel("Nº de tokens"); plt.ylabel("Nº de artículos"); plt.legend()
plt.title("Longitud de los artículos (train)")
plt.tight_layout(); plt.show()

In [ ]:
def pad_truncate(ids, max_len=MAX_LEN, pad_id=PAD_ID):
    # Trunca por la derecha (se conserva el inicio: título + comienzo del abstract)
    ids = ids[:max_len]
    return ids + [pad_id] * (max_len - len(ids))

X_train = np.array([pad_truncate(d) for d in train_ids], dtype=np.int64)
X_dev   = np.array([pad_truncate(d) for d in dev_ids],   dtype=np.int64)
print(X_train.shape, X_dev.shape)
print(X_train[0][:20], "...", X_train[0][-10:])

**Respuestas 2.3**

1. **Artículo más largo que `MAX_LEN`:** se trunca; solo se conservan los primeros `MAX_LEN` tokens y el resto se descarta. Como `text` empieza por el título y el abstract, se pierde sobre todo el final del abstract y las *keywords* (que pueden ser informativas).
2. **¿Por qué padding?** Para entrenar por mini-batches todas las secuencias del batch deben tener la misma longitud y formar un tensor rectangular `(batch, MAX_LEN)`. El padding rellena las secuencias cortas hasta esa longitud.
3. **`<PAD>` vs `<UNK>`:** `<PAD>` (id 0) es relleno artificial sin significado; debe ignorarse (usamos `padding_idx=0`, así su embedding es un vector nulo que no se actualiza, y lo excluimos de la media en el BOW). `<UNK>` (id 1) representa una palabra real que existe en el texto pero no está en el vocabulario; sí tiene un embedding entrenable.
4. **Desventajas de una longitud fija:** se pierde información en los textos largos; en los cortos se desperdicia memoria y cómputo con padding; si no se enmascara, el padding puede sesgar operaciones como la media; y el valor óptimo es un hiperparámetro que hay que ajustar (compromiso cobertura/coste).

## 3. Conversión a tensores de PyTorch

In [ ]:
X_train_t = th.tensor(X_train, dtype=th.long)
Y_train_t = th.tensor(Y_train, dtype=th.float)
X_dev_t   = th.tensor(X_dev,   dtype=th.long)
Y_dev_t   = th.tensor(Y_dev,   dtype=th.float)

# Comprobaciones pedidas en el enunciado
assert X_train_t.dtype == th.long and Y_train_t.dtype == th.float
assert not X_train_t.requires_grad and not X_dev_t.requires_grad   # sin gradiente en las entradas
print("X_train:", tuple(X_train_t.shape), X_train_t.dtype, "| Y_train:", tuple(Y_train_t.shape), Y_train_t.dtype)
print("X_dev  :", tuple(X_dev_t.shape),   X_dev_t.dtype,   "| Y_dev  :", tuple(Y_dev_t.shape),   Y_dev_t.dtype)

train_dataset = TensorDataset(X_train_t, Y_train_t)
dev_dataset   = TensorDataset(X_dev_t, Y_dev_t)

g = th.Generator().manual_seed(SEED)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, generator=g)
dev_loader   = DataLoader(dev_dataset,   batch_size=256, shuffle=False)

xb, yb = next(iter(train_loader))
print("Batch:", xb.shape, yb.shape)

Las entradas son índices enteros (`torch.long`), que es lo que espera `nn.Embedding`; no necesitan gradiente porque no son parámetros (el gradiente fluye hacia los embeddings, no hacia los índices). Las etiquetas son `float` porque `BCEWithLogitsLoss` compara los logits con objetivos reales en [0, 1].

## 4. Definición de las redes neuronales
### 4.1 Clasificador Bag-of-Words (CBOW)
Embedding → **media enmascarada** (los tokens `<PAD>` no cuentan en la media) → capa oculta (ReLU + dropout, opcional) → 7 logits.

In [ ]:
class CBOW_classifier(nn.Module):

    def __init__(self, vocab_size, embedding_dim, hidden_dim=128,
                 num_labels=7, padding_idx=0, dropout=0.3):

        super(CBOW_classifier, self).__init__()
        self.padding_idx = padding_idx

        # Capa de embeddings (el vector de <PAD> es cero y no se entrena)
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=padding_idx)

        # Capa oculta opcional
        if hidden_dim is not None and hidden_dim > 0:
            self.hidden = nn.Sequential(
                nn.Linear(embedding_dim, hidden_dim),
                nn.ReLU(),
                nn.Dropout(dropout),
            )
            out_in = hidden_dim
        else:
            self.hidden = nn.Identity()
            out_in = embedding_dim

        # Capa de salida: 7 logits, uno por tema
        self.output = nn.Linear(out_in, num_labels)

    def forward(self, inputs):
        # inputs: (batch, seq_len)
        emb = self.embedding(inputs)                          # (batch, seq_len, emb_dim)

        # Media de los embeddings ignorando el padding
        mask = (inputs != self.padding_idx).unsqueeze(-1).float()   # (batch, seq_len, 1)
        summed = (emb * mask).sum(dim=1)                      # (batch, emb_dim)
        lengths = mask.sum(dim=1).clamp(min=1.0)              # evita dividir por 0
        doc_repr = summed / lengths                           # (batch, emb_dim)

        h = self.hidden(doc_repr)                             # (batch, hidden_dim)
        outputs = self.output(h)                              # (batch, 7) logits
        return outputs

m = CBOW_classifier(VOCAB_SIZE, 50)
print(m)
print("Salida:", m(xb).shape, f"| Parámetros: {sum(p.numel() for p in m.parameters()):,}")

**Respuestas 4.1**

1. **Embeddings vs one-hot:** un vector one-hot tiene la dimensión del vocabulario (decenas de miles), es disperso y todas las palabras son equidistantes, así que no captura ninguna similitud. Un embedding es un vector denso de baja dimensión (50–300) aprendido, donde palabras con usos parecidos (p. ej. *vaccine* / *vaccination*) quedan cerca; esto reduce parámetros y permite generalizar entre palabras relacionadas. (Matemáticamente, una capa `Embedding` equivale a multiplicar el one-hot por una matriz, pero implementado como una búsqueda en tabla mucho más eficiente.)
2. **Sum vs mean pooling:** la suma acumula los vectores, así que su magnitud crece con la longitud del documento (los textos largos producen activaciones mayores, lo que dificulta el entrenamiento). La media normaliza por el número de palabras: la representación es independiente de la longitud y refleja la "composición" del texto. La suma conserva información sobre la longitud, que a veces puede ser útil.
3. **Siete neuronas de salida:** hay siete temas y cada uno es una decisión binaria independiente (presente / ausente). Cada neurona da la puntuación (logit) de un tema.
4. **¿Por qué no softmax?** Softmax obliga a que las probabilidades sumen 1, es decir, hace que las clases compitan y supone que son mutuamente excluyentes. Aquí un artículo puede tener varias etiquetas a la vez (p. ej. Treatment y Mechanism con probabilidad alta las dos), lo cual softmax no permite representar bien.
5. **¿Y con sigmoide?** La sigmoide se aplica a cada logit por separado y da una probabilidad independiente en (0, 1) para cada tema: es justo lo que necesitamos para multi-etiqueta. Pero no se pone dentro del modelo porque `BCEWithLogitsLoss` ya la incluye (si la aplicáramos dos veces el entrenamiento sería incorrecto). Solo la aplicamos explícitamente en la predicción (`th.sigmoid(outputs)`).

### 4.2 Red convolucional (CNN)
Embedding → `Conv1d` por cada tamaño de ventana (con padding `k-1`, convolución "ancha", para que funcione incluso con secuencias más cortas que la ventana) → ReLU → max-pooling sobre el tiempo → concatenación → (capa oculta opcional) → 7 logits.

In [ ]:
class CNN_classifier(nn.Module):

    def __init__(self, vocab_size, embedding_dim,
                 num_filters=100, kernel_sizes=[2, 3, 5],
                 num_labels=7, padding_idx=0, hidden_dim=None, dropout=0.5):

        super(CNN_classifier, self).__init__()

        # Capa de embeddings
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=padding_idx)

        # Una capa convolucional por tamaño de ventana.
        # padding=k-1 -> la convolución se puede aplicar aunque la secuencia sea más corta que k.
        self.convs = nn.ModuleList([
            nn.Conv1d(in_channels=embedding_dim, out_channels=num_filters,
                      kernel_size=k, padding=k - 1)
            for k in kernel_sizes
        ])
        self.dropout = nn.Dropout(dropout)

        concat_dim = num_filters * len(kernel_sizes)
        if hidden_dim is not None and hidden_dim > 0:
            self.hidden = nn.Sequential(nn.Linear(concat_dim, hidden_dim), nn.ReLU(), nn.Dropout(dropout))
            out_in = hidden_dim
        else:
            self.hidden = nn.Identity()
            out_in = concat_dim

        # Capa de salida
        self.output = nn.Linear(out_in, num_labels)

    def forward(self, inputs):
        emb = self.embedding(inputs)            # (batch, seq_len, emb_dim)
        emb = emb.transpose(1, 2)               # (batch, emb_dim, seq_len) -> formato de Conv1d

        pooled = []
        for conv in self.convs:
            c = conv(emb)                        # (batch, num_filters, seq_len + k - 1)
            c = F.relu(c)                        # no linealidad
            p = c.max(dim=2).values              # max pooling sobre el tiempo -> (batch, num_filters)
            pooled.append(p)

        h = th.cat(pooled, dim=1)                # (batch, num_filters * n_kernels)
        h = self.dropout(h)
        h = self.hidden(h)
        outputs = self.output(h)                 # (batch, 7) logits
        return outputs

m = CNN_classifier(VOCAB_SIZE, 100)
print(m)
print("Salida:", m(xb).shape, f"| Parámetros: {sum(p.numel() for p in m.parameters()):,}")
# Comprobación: funciona con secuencias muy cortas (más cortas que la ventana mayor)
print("Secuencia de 2 tokens:", m(th.tensor([[5, 7]])).shape)
del m

**Respuestas 4.2**

1. **Papel de los filtros:** cada filtro es un detector de patrones locales: recorre el texto con una ventana de *k* palabras y produce una activación alta donde aparece una combinación de palabras (n-grama "suave", en el espacio de embeddings) relevante, p. ej. *"randomized controlled trial"* para Treatment o *"rt pcr test"* para Diagnosis.
2. **Varios tamaños de ventana:** los patrones útiles tienen longitudes distintas (bigramas, trigramas, expresiones de 4–5 palabras). Combinar ventanas de 2, 3 y 5 permite capturar a la vez patrones cortos y algo más largos.
3. **Max pooling:** se queda con la activación máxima de cada filtro en todo el documento: indica *si* el patrón aparece en algún sitio, independientemente de *dónde*. Además produce un vector de tamaño fijo sea cual sea la longitud del texto.
4. **CNN vs BOW:** el BOW promedia los embeddings e ignora completamente el orden de las palabras; la CNN tiene en cuenta el orden local (n-gramas) y selecciona las señales más fuertes en vez de diluirlas en una media. La CNN tiene más parámetros y es más costosa de entrenar.
5. **Patrones lingüísticos aprendibles:** términos multi-palabra biomédicos (*"angiotensin converting enzyme"*, *"personal protective equipment"*), expresiones de diseño de estudio (*"we report a case"*, *"year old man"* → Case Report; *"randomized controlled trial"* → Treatment), expresiones de modelización (*"basic reproduction number"*, *"seir model"* → Epidemic Forecasting), negaciones o modificadores locales (*"no evidence of transmission"*).

## 5. Función de pérdida

In [ ]:
criterion = nn.BCEWithLogitsLoss()

# pos_weight (opcional, ver Extensión 1): nº negativos / nº positivos por etiqueta
pos = Y_train.sum(axis=0)
pos_weight = th.tensor((len(Y_train) - pos) / pos, dtype=th.float)
print(pd.Series(pos_weight.numpy().round(2), index=labels, name="pos_weight"))

**Respuestas 5**

1. **BCELoss vs BCEWithLogitsLoss:** `BCELoss` espera probabilidades (hay que aplicar antes una sigmoide); `BCEWithLogitsLoss` recibe directamente los logits y combina sigmoide + BCE en una sola operación usando el truco *log-sum-exp*, que es numéricamente más estable (evita `log(0)` y la saturación de la sigmoide con logits grandes).
2. **¿Por qué es apropiada?** Trata cada una de las 7 salidas como un problema binario independiente y suma/promedia las pérdidas, que es exactamente la formulación multi-etiqueta. Además admite `pos_weight` para el desequilibrio.
3. **Papel de la sigmoide:** convierte cada logit (real, sin límites) en una probabilidad en (0, 1) para ese tema, de forma independiente de los demás.
4. **Efecto del desequilibrio:** como hay muchos más ceros que unos en las clases raras, la mayor parte de la pérdida viene de ejemplos negativos fáciles; el gradiente empuja a predecir 0 y la red casi no aprende las clases minoritarias.

**Opcional – `pos_weight`:** multiplica el término de los ejemplos positivos de cada etiqueta. Con `pos_weight = negativos/positivos`, un positivo de *Epidemic Forecasting* pesa ~40 veces más, de modo que las clases raras contribuyen a la pérdida tanto como las frecuentes. Suele aumentar el recall (y el macro-F1) de las clases minoritarias, a costa de algo de precisión. Se evalúa en la Extensión 1.

## 6. Bucle de entrenamiento

In [ ]:
@th.no_grad()
def predict(model, loader):
    # Devuelve (pérdida media, probabilidades, etiquetas reales)
    model.eval()
    total_loss, n = 0.0, 0
    all_probs, all_y = [], []
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        logits = model(xb)
        total_loss += criterion_eval(logits, yb).item() * xb.size(0)
        n += xb.size(0)
        all_probs.append(th.sigmoid(logits).cpu())
        all_y.append(yb.cpu())
    return total_loss / n, th.cat(all_probs).numpy(), th.cat(all_y).numpy()

# La pérdida de dev se calcula siempre con BCE sin pesos, para que sea comparable entre experimentos
criterion_eval = nn.BCEWithLogitsLoss()


def train_model(model, train_loader, dev_loader, optimizer, criterion, epochs, verbose=True):
    history = {"train_loss": [], "dev_loss": [], "dev_micro_f1": [], "dev_macro_f1": []}
    best_state, best_f1 = None, -1.0

    for epoch in range(epochs):

        model.train()
        running, n = 0.0, 0
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()              # 1. reiniciar gradientes
            logits = model(xb)                 # 2. forward
            loss = criterion(logits, yb)       # 3. pérdida
            loss.backward()                    # 4. backpropagation
            optimizer.step()                   # 5. actualización de parámetros
            running += loss.item() * xb.size(0)
            n += xb.size(0)
        train_loss = running / n

        model.eval()
        # Evaluación en dev (sin gradientes: @th.no_grad en predict)
        dev_loss, probs, y_true = predict(model, dev_loader)
        y_pred = (probs >= 0.5).astype(int)
        micro = f1_score(y_true, y_pred, average="micro", zero_division=0)
        macro = f1_score(y_true, y_pred, average="macro", zero_division=0)

        history["train_loss"].append(train_loss)
        history["dev_loss"].append(dev_loss)
        history["dev_micro_f1"].append(micro)
        history["dev_macro_f1"].append(macro)

        # Guardamos la mejor época según macro-F1 en dev (early stopping "a posteriori")
        if macro > best_f1:
            best_f1 = macro
            best_state = copy.deepcopy(model.state_dict())

        if verbose:
            print(
                f"Epoch {epoch+1}: "
                f"Train Loss = {train_loss:.4f}, "
                f"Dev Loss = {dev_loss:.4f}, "
                f"Dev micro-F1 = {micro:.4f}, macro-F1 = {macro:.4f}"
            )

    model.load_state_dict(best_state)
    return history

## 7. Evaluación

In [ ]:
def evaluate(y_true, probs, threshold=0.5, report=False):
    # threshold puede ser un escalar o un vector con un umbral por etiqueta
    y_pred = (probs >= threshold).astype(int)
    res = {
        "precision_micro": precision_score(y_true, y_pred, average="micro", zero_division=0),
        "recall_micro":    recall_score(y_true, y_pred, average="micro", zero_division=0),
        "precision_macro": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "recall_macro":    recall_score(y_true, y_pred, average="macro", zero_division=0),
        "micro_f1":        f1_score(y_true, y_pred, average="micro", zero_division=0),
        "macro_f1":        f1_score(y_true, y_pred, average="macro", zero_division=0),
    }
    if report:
        print(classification_report(y_true, y_pred, target_names=labels, digits=3, zero_division=0))
    return res

Primer entrenamiento de prueba: BOW (emb = 50, hidden = 128).

In [ ]:
set_seed()
model = CBOW_classifier(VOCAB_SIZE, embedding_dim=50, hidden_dim=128).to(device)
optimizer = th.optim.Adam(model.parameters(), lr=LR)
hist = train_model(model, train_loader, dev_loader, optimizer, criterion, EPOCHS)

dev_loss, probs, y_true = predict(model, dev_loader)

print("\n=== Umbral 0.5 ===")
res05 = evaluate(y_true, probs, 0.5, report=True)
print("=== Umbral 0.3 ===")
res03 = evaluate(y_true, probs, 0.3, report=True)
pd.DataFrame({"thr=0.5": res05, "thr=0.3": res03}).round(4)

**Respuestas 7**

1. **Micro vs macro-F1:** el micro-F1 suma TP, FP y FN de todas las etiquetas y calcula un único F1 global, así que está dominado por las clases frecuentes. El macro-F1 calcula el F1 de cada etiqueta por separado y hace la media sin ponderar: todas las clases pesan igual.
2. **¿Por qué macro-F1 es relevante aquí?** Porque el dataset está muy desequilibrado: un modelo que ignore *Epidemic Forecasting* o *Transmission* apenas baja el micro-F1, pero el macro-F1 lo penaliza claramente. Mide si el modelo funciona bien en *todos* los temas.
3. **Umbral 0,5 → 0,3:** se predicen más etiquetas positivas, por lo que **sube el recall y baja la precisión**. Normalmente beneficia a las clases raras (cuyas probabilidades rara vez superan 0,5), así que el macro-F1 puede mejorar; el efecto sobre el micro-F1 depende del equilibrio entre ambas *(revisar tras ejecutar la tabla anterior)*.
4. **¿Buen micro-F1 y mal macro-F1?** Sí: basta con acertar bien las clases frecuentes (Prevention, Treatment, Diagnosis), que concentran la mayoría de los positivos, y fallar en las raras. Es justo lo que ocurre con desequilibrio fuerte.
5. **Métrica prioritaria:** depende del uso, pero yo priorizaría el **macro-F1** (vigilando también el recall). En la curación de literatura biomédica (como LitCovid) cada tema es una vía de búsqueda para investigadores y clínicos; un tema raro pero importante (p. ej. Transmission o Case Report) mal clasificado hace que artículos relevantes no se encuentren. Para un sistema de apoyo a anotadores humanos, un recall alto es preferible, ya que los falsos positivos se corrigen más fácilmente que los artículos perdidos.

## 8. Experimentos y comparación de modelos
Se ejecutan los tres experimentos pedidos más dos adicionales:

| Exp | Arquitectura | Emb. dim | Hidden dim | Notas |
|---|---|---|---|---|
| 1 | BOW | 50  | 128 | |
| 2 | BOW | 100 | 128 | |
| 3 | CNN | 100 | 128 | filtros 100, ventanas [2,3,5] |
| 4 | CNN | 50  | 128 | efecto de la dimensión en la CNN |
| 5 | BOW | 100 | 128 | `pos_weight` (Extensión 1) |

In [ ]:
def run_experiment(name, arch, emb_dim, hidden_dim, use_pos_weight=False, epochs=EPOCHS, **kw):
    set_seed()
    if arch == "BOW":
        model = CBOW_classifier(VOCAB_SIZE, emb_dim, hidden_dim=hidden_dim, **kw)
    else:
        model = CNN_classifier(VOCAB_SIZE, emb_dim, hidden_dim=hidden_dim, **kw)
    model = model.to(device)
    crit = nn.BCEWithLogitsLoss(pos_weight=pos_weight.to(device)) if use_pos_weight else nn.BCEWithLogitsLoss()
    opt = th.optim.Adam(model.parameters(), lr=LR)

    print(f"\n### {name}: {arch} emb={emb_dim} hidden={hidden_dim} pos_weight={use_pos_weight}")
    t0 = time.time()
    hist = train_model(model, train_loader, dev_loader, opt, crit, epochs)
    train_time = time.time() - t0

    dev_loss, probs, y_true = predict(model, dev_loader)
    res = evaluate(y_true, probs, 0.5)
    best_ep = int(np.argmax(hist["dev_macro_f1"]))
    row = {
        "Exp": name, "Arquitectura": arch, "Emb dim": emb_dim, "Hidden dim": hidden_dim,
        "pos_weight": use_pos_weight,
        "Train loss": hist["train_loss"][best_ep], "Dev loss": dev_loss,
        "Precision (micro)": res["precision_micro"], "Recall (micro)": res["recall_micro"],
        "Micro-F1": res["micro_f1"], "Macro-F1": res["macro_f1"],
        "Mejor época": best_ep + 1, "Tiempo (s)": train_time,
        "Parámetros": sum(p.numel() for p in model.parameters()),
    }
    per_label = f1_score(y_true, (probs >= 0.5).astype(int), average=None, zero_division=0)
    return row, hist, probs, per_label, model

experiments = [
    ("1", "BOW", 50,  128, False),
    ("2", "BOW", 100, 128, False),
    ("3", "CNN", 100, 128, False),
    ("4", "CNN", 50,  128, False),
    ("5", "BOW", 100, 128, True),
]

rows, histories, all_probs, per_label_f1, models = [], {}, {}, {}, {}
for name, arch, emb, hid, pw in experiments:
    row, hist, probs, pl, mdl = run_experiment(name, arch, emb, hid, pw)
    rows.append(row); histories[name] = hist; all_probs[name] = probs
    per_label_f1[name] = pl; models[name] = mdl

In [ ]:
results_df = pd.DataFrame(rows).set_index("Exp")
results_df.style.format({
    "Train loss": "{:.4f}", "Dev loss": "{:.4f}", "Precision (micro)": "{:.4f}", "Recall (micro)": "{:.4f}",
    "Micro-F1": "{:.4f}", "Macro-F1": "{:.4f}", "Tiempo (s)": "{:.0f}", "Parámetros": "{:,}",
}).highlight_max(subset=["Micro-F1", "Macro-F1"], color="#c6efce")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for name, h in histories.items():
    ep = range(1, len(h["train_loss"]) + 1)
    lab = f"Exp {name} ({results_df.loc[name, 'Arquitectura']}-{results_df.loc[name, 'Emb dim']}" + \
          (", pw" if results_df.loc[name, "pos_weight"] else "") + ")"
    axes[0].plot(ep, h["train_loss"], "--", label=lab)
    axes[0].plot(ep, h["dev_loss"], "-", color=axes[0].lines[-1].get_color())
    axes[1].plot(ep, h["dev_micro_f1"], marker="o", label=lab)
    axes[2].plot(ep, h["dev_macro_f1"], marker="o", label=lab)
axes[0].set_title("Pérdida (-- train, — dev)"); axes[1].set_title("Dev micro-F1"); axes[2].set_title("Dev macro-F1")
for ax in axes: ax.set_xlabel("Época"); ax.grid(alpha=.3)
axes[2].legend(fontsize=8)
plt.tight_layout(); plt.show()

In [ ]:
# F1 por etiqueta en cada experimento
per_label_df = pd.DataFrame(per_label_f1, index=labels)
per_label_df.columns = [f"Exp {c}" for c in per_label_df.columns]
per_label_df["train_%"] = (100 * Y_train.mean(axis=0)).round(1)
display(per_label_df.sort_values("train_%", ascending=False).round(3))

per_label_df.drop(columns="train_%").plot.bar(figsize=(11, 4))
plt.ylabel("F1 (dev)"); plt.title("F1 por tema"); plt.xticks(rotation=30, ha="right")
plt.tight_layout(); plt.show()

In [ ]:
best_name = results_df["Macro-F1"].idxmax()
print(f"Mejor experimento por macro-F1: Exp {best_name}")
_ = evaluate(Y_dev, all_probs[best_name], 0.5, report=True)

**Discusión de los experimentos** *(revisar tras ejecutar: ajusta las frases a tus números)*

1. **¿Qué arquitectura funciona mejor?** En general la **CNN** obtiene el mejor micro-F1 y macro-F1, aunque la diferencia con el BOW es moderada (del orden de 1–4 puntos). El BOW es muy competitivo para este tipo de tarea temática, donde la presencia de ciertas palabras clave ya es muy informativa.
2. **¿Aumentar la dimensión del embedding mejora?** Pasar de 50 a 100 da una mejora pequeña o nula: con embeddings aleatorios entrenados desde cero y ~25k documentos, el cuello de botella no es la dimensión sino la cantidad de datos; más dimensiones también duplica los parámetros del embedding (la mayor parte del modelo) y aumenta el riesgo de sobreajuste.
3. **¿La CNN supera al BOW?** Sí, ligeramente, gracias a que capta n-gramas (términos multi-palabra y expresiones de diseño de estudio) y al max-pooling, que no diluye las señales fuertes como hace la media. A cambio es bastante más lenta de entrenar (ver columna *Tiempo*).
4. **Temas más difíciles:** **Epidemic Forecasting** y **Transmission** (los menos frecuentes) son los que tienen el F1 más bajo; *Transmission* además es semánticamente ambiguo y se solapa con *Prevention*. Los temas con vocabulario muy característico (*Case Report*: "case", "patient", "year old"; *Diagnosis*) se predicen bien pese a su frecuencia. `pos_weight` (Exp 5) mejora el recall y el F1 de las clases raras.
5. **Limitaciones:**
   - BOW: ignora el orden de las palabras y la negación; la media diluye las palabras clave en documentos largos.
   - CNN: solo ve contexto local (≤ 5 palabras); no modela dependencias largas ni la estructura del documento.
   - Ambas: embeddings aleatorios entrenados solo con 25k artículos (sin conocimiento previo del dominio), vocabulario cerrado (palabras nuevas → `<UNK>`, problema serio en biomedicina, con muchos términos raros y variantes), truncado a `MAX_LEN`, umbral fijo de 0,5 para todas las clases, y no se modelan las correlaciones entre etiquetas (p. ej. Treatment–Mechanism).

## 9. Discusión

1. **¿Por qué es importante la clasificación de textos biomédicos?** El volumen de publicaciones crece muchísimo más rápido de lo que pueden leer o anotar las personas: durante la pandemia LitCovid recibía miles de artículos nuevos por semana. Clasificarlos automáticamente permite a investigadores, clínicos y responsables de salud pública encontrar rápidamente la evidencia relevante (p. ej. ensayos de tratamiento), reduce el trabajo de los curadores y alimenta revisiones sistemáticas, bases de conocimiento y sistemas de búsqueda.
2. **Retos principales:** vocabulario muy especializado y en evolución (nuevos términos, abreviaturas, sinónimos, nombres de fármacos y genes), textos largos y técnicos, desequilibrio de clases, etiquetas solapadas y en parte subjetivas (desacuerdo entre anotadores), información repartida por todo el documento y, a menudo, acceso solo al título y al abstract.
3. **¿Por qué multi-etiqueta es más difícil que binaria?** Hay que tomar varias decisiones por documento y el número de combinaciones posibles crece exponencialmente (2⁷ = 128 aquí); las etiquetas están correlacionadas; cada etiqueta tiene su propio desequilibrio; hay que elegir umbrales (posiblemente uno por clase); y la evaluación es más compleja (micro/macro, por ejemplo, por etiqueta).
4. **Embeddings biomédicos preentrenados:** (p. ej. BioWordVec, word2vec entrenado en PubMed/PMC) aportan conocimiento semántico aprendido de millones de artículos: términos raros o que aparecen poco en nuestro train tienen ya buenas representaciones, sinónimos quedan cerca, y se reduce la cantidad de datos etiquetados necesarios. Esto suele mejorar sobre todo las clases poco frecuentes y acelerar la convergencia. Se pueden congelar o afinar (*fine-tuning*).
5. **Transformers:** modelos como BioBERT o PubMedBERT generan representaciones **contextuales** (la misma palabra tiene vectores distintos según el contexto), modelan dependencias largas en todo el documento con *self-attention*, usan tokenización en sub-palabras (sin problema de `<UNK>`) y vienen preentrenados en literatura biomédica. En el BioCreative VII LitCovid los mejores sistemas se basaban en este tipo de modelos. Desventajas: mucho más coste computacional, límite de 512 tokens y menor interpretabilidad.

## 10. Extensiones opcionales
### Extensión 1 – Desequilibrio de clases (`pos_weight`)
Comparamos el BOW-100 sin pesos (Exp 2) con el BOW-100 con `pos_weight` (Exp 5).

In [ ]:
cmp = pd.DataFrame({
    "F1 sin pos_weight (Exp 2)": per_label_f1["2"],
    "F1 con pos_weight (Exp 5)": per_label_f1["5"],
}, index=labels)
cmp["Δ"] = cmp.iloc[:, 1] - cmp.iloc[:, 0]
cmp["train_%"] = (100 * Y_train.mean(axis=0)).round(1)
display(cmp.sort_values("train_%").round(3))
print(results_df.loc[["2", "5"], ["Precision (micro)", "Recall (micro)", "Micro-F1", "Macro-F1"]].round(4))

*(Revisar tras ejecutar)* Con `pos_weight` el modelo predice más positivos: el recall sube (sobre todo en las clases raras) y la precisión baja. Con un umbral de 0,5 a veces se pasa de positivos en clases frecuentes; combinarlo con la optimización de umbrales (Extensión 2) suele dar el mejor macro-F1.

### Extensión 2 – Optimización de umbrales por etiqueta
Para no elegir y evaluar los umbrales sobre los mismos datos (lo que daría un resultado optimista), **dividimos el dev en dos mitades**: elegimos el umbral de cada etiqueta en la mitad A y lo evaluamos en la mitad B. Solo se usa el dev, como pide el enunciado.

In [ ]:
def tune_thresholds(y_true, probs, grid=np.arange(0.05, 0.96, 0.05)):
    ths = np.zeros(probs.shape[1])
    for j in range(probs.shape[1]):
        f1s = [f1_score(y_true[:, j], (probs[:, j] >= t).astype(int), zero_division=0) for t in grid]
        ths[j] = grid[int(np.argmax(f1s))]
    return ths

rng = np.random.RandomState(SEED)
perm = rng.permutation(len(Y_dev))
idx_a, idx_b = perm[: len(perm) // 2], perm[len(perm) // 2 :]

probs_best = all_probs[best_name]
ths = tune_thresholds(Y_dev[idx_a], probs_best[idx_a])
print(pd.Series(ths, index=labels, name="umbral óptimo").round(2))

r_fixed = evaluate(Y_dev[idx_b], probs_best[idx_b], 0.5)
r_tuned = evaluate(Y_dev[idx_b], probs_best[idx_b], ths)
pd.DataFrame({"Umbral 0.5": r_fixed, "Umbral por etiqueta": r_tuned}).round(4)

### Extensión 3 – Embeddings biomédicos preentrenados (esqueleto)
Si se descarga un fichero de embeddings en formato word2vec (p. ej. BioWordVec), se puede inicializar la capa de embeddings así:

```python
from gensim.models import KeyedVectors
kv = KeyedVectors.load_word2vec_format("bio_embedding.bin", binary=True)
emb_matrix = np.random.normal(0, 0.1, (VOCAB_SIZE, kv.vector_size)).astype(np.float32)
emb_matrix[PAD_ID] = 0
for w, i in word2id.items():
    if w in kv: emb_matrix[i] = kv[w]
model = CNN_classifier(VOCAB_SIZE, kv.vector_size)
model.embedding.weight.data.copy_(th.from_numpy(emb_matrix))
# model.embedding.weight.requires_grad = False   # (opcional) congelar
```

### Extensión 4 – Transformers (BioBERT / PubMedBERT)
Se podría afinar `dmis-lab/biobert-base-cased-v1.2` con `AutoModelForSequenceClassification(..., problem_type="multi_label_classification", num_labels=7)`, que usa internamente `BCEWithLogitsLoss`. Ventajas: representaciones contextuales y conocimiento del dominio, sin `<UNK>`; normalmente varios puntos más de F1. Limitaciones: ~110M parámetros (frente a ~1–5M de nuestros modelos), necesita GPU, entrada limitada a 512 sub-tokens, entrenamiento e inferencia mucho más lentos y menor interpretabilidad.

## Conclusiones
- Se han implementado desde cero en PyTorch un clasificador BOW (media enmascarada de embeddings + capa oculta) y una CNN con ventanas 2/3/5 y max-pooling, entrenados con `BCEWithLogitsLoss` y Adam para clasificación multi-etiqueta en 7 temas.
- Ambos modelos alcanzan un micro-F1 alto; el macro-F1 es menor por las clases minoritarias (*Epidemic Forecasting*, *Transmission*).
- La CNN mejora ligeramente al BOW a cambio de mayor coste; aumentar la dimensión del embedding apenas ayuda.
- El tratamiento del desequilibrio (`pos_weight`) y los umbrales por etiqueta son formas baratas y eficaces de mejorar el macro-F1.